In [1]:
import pandas as pd
import numpy as np
import joblib

train_df = joblib.load('../data/processed/train_processed.pkl')
val_df = joblib.load('../data/processed/val_processed.pkl')

split_point = len(train_df)

full_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
full_df = full_df.sort_values('TransactionDT').reset_index(drop=True)

print(f"train_df: {train_df.shape}, val_df: {val_df.shape}")
print(f"full_df: {full_df.shape}, split_point: {split_point}")
print(f"\nIs already sorted by time (should be True): {full_df['TransactionDT'].is_monotonic_increasing}")

train_df: (472432, 434), val_df: (118108, 434)
full_df: (590540, 434), split_point: 472432

Is already sorted by time (should be True): True


In [2]:
full_df['card_entity'] = (
    full_df['card1'].astype(str) + '_' +
    full_df['card2'].astype(str) + '_' +
    full_df['addr1'].astype(str)
)

print("Unique card_entity values:", full_df['card_entity'].nunique())
print("\nTransactions per entity (distribution):")
print(full_df['card_entity'].value_counts().describe())

Unique card_entity values: 41315

Transactions per entity (distribution):
count    41315.000000
mean        14.293598
std        104.064667
min          1.000000
25%          1.000000
50%          2.000000
75%          7.000000
max       9900.000000
Name: count, dtype: float64


/tmp/ipykernel_1974/662742873.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['card_entity'] = (


In [3]:
full_df = full_df.sort_values(['card_entity', 'TransactionDT']).reset_index(drop=True)

# Count of prior transactions by this entity (expanding, strictly before current row)
full_df['entity_txn_count_so_far'] = full_df.groupby('card_entity').cumcount()

# Time since this entity's previous transaction (seconds)
full_df['time_since_last_txn'] = full_df.groupby('card_entity')['TransactionDT'].diff()

print(full_df[['card_entity', 'TransactionDT', 'entity_txn_count_so_far', 'time_since_last_txn']].head(10))

         card_entity  TransactionDT  entity_txn_count_so_far  \
0  10000_111.0_184.0        4050851                        0   
1  10003_361.0_299.0        7840676                        0   
2  10003_555.0_299.0        8421815                        0   
3  10003_555.0_299.0        8634882                        1   
4  10003_555.0_299.0        8635215                        2   
5  10003_555.0_299.0        8642405                        3   
6  10004_529.0_123.0        3271265                        0   
7  10004_529.0_177.0         663069                        0   
8  10004_529.0_177.0        8345697                        1   
9  10004_529.0_191.0        1454318                        0   

   time_since_last_txn  
0                  NaN  
1                  NaN  
2                  NaN  
3             213067.0  
4                333.0  
5               7190.0  
6                  NaN  
7                  NaN  
8            7682628.0  
9                  NaN  


/tmp/ipykernel_1974/2569970834.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['entity_txn_count_so_far'] = full_df.groupby('card_entity').cumcount()
/tmp/ipykernel_1974/2569970834.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['time_since_last_txn'] = full_df.groupby('card_entity')['TransactionDT'].diff()


In [4]:
full_df['time_since_last_txn'] = full_df['time_since_last_txn'].fillna(999999)

def rolling_window_features(group, window_seconds, col_name_prefix):
    times = group['TransactionDT'].values
    amounts = group['TransactionAmt'].values
    counts = np.zeros(len(group))
    avg_amounts = np.zeros(len(group))
    
    for i in range(len(group)):
        window_start = times[i] - window_seconds
        mask = (times[:i] >= window_start) & (times[:i] < times[i])
        counts[i] = mask.sum()
        avg_amounts[i] = amounts[:i][mask].mean() if mask.sum() > 0 else 0
    
    return pd.DataFrame({
        f'{col_name_prefix}_count': counts,
        f'{col_name_prefix}_avg_amt': avg_amounts
    }, index=group.index)

print("This will take a few minutes on the full dataset - testing on a sample first")

sample_entities = full_df['card_entity'].value_counts().head(100).index
sample_df = full_df[full_df['card_entity'].isin(sample_entities)].copy()

result = sample_df.groupby('card_entity', group_keys=False).apply(
    lambda g: rolling_window_features(g, 3600, 'txn_1h')
)
print(result.head(10))
print(f"\nSample processed: {len(sample_df)} rows")

This will take a few minutes on the full dataset - testing on a sample first
     txn_1h_count  txn_1h_avg_amt
533           0.0            0.00
534           1.0          171.00
535           2.0          100.50
536           0.0            0.00
537           0.0            0.00
538           0.0            0.00
539           1.0           40.00
540           0.0            0.00
541           0.0            0.00
542           1.0           57.95

Sample processed: 149945 rows


In [5]:
import time

start = time.time()
result_test = sample_df.groupby('card_entity', group_keys=False).apply(
    lambda g: rolling_window_features(g, 3600, 'txn_1h')
)
elapsed = time.time() - start
print(f"Time for 100 entities ({len(sample_df)} rows): {elapsed:.1f}s")
print(f"Estimated time for all 41,315 entities: {elapsed * (41315/100) / 60:.1f} minutes")

Time for 100 entities (149945 rows): 1.5s
Estimated time for all 41,315 entities: 10.1 minutes


In [6]:
def compute_rolling_features_vectorized(df, window_seconds, prefix):
    df = df.sort_values(['card_entity', 'TransactionDT']).copy()
    df['_dt'] = pd.to_datetime(df['TransactionDT'], unit='s')
    df = df.set_index('_dt')
    
    grouped = df.groupby('card_entity')
    
    # Rolling count of transactions in the trailing window (including current row, so subtract 1)
    rolling_count = grouped['TransactionAmt'].rolling(f'{window_seconds}s').count()
    rolling_sum = grouped['TransactionAmt'].rolling(f'{window_seconds}s').sum()
    
    df[f'{prefix}_count'] = rolling_count.values - 1  # exclude current transaction
    df[f'{prefix}_sum_prior'] = rolling_sum.values - df['TransactionAmt'].values
    df[f'{prefix}_avg_amt'] = np.where(
        df[f'{prefix}_count'] > 0,
        df[f'{prefix}_sum_prior'] / df[f'{prefix}_count'],
        0
    )
    
    df = df.reset_index(drop=True)
    return df[[f'{prefix}_count', f'{prefix}_avg_amt']]

start = time.time()
test_result = compute_rolling_features_vectorized(sample_df, 3600, 'txn_1h')
elapsed = time.time() - start
print(f"Vectorized time for 100 entities: {elapsed:.2f}s")
print(test_result.head(10))

Vectorized time for 100 entities: 0.15s
   txn_1h_count  txn_1h_avg_amt
0           0.0            0.00
1           1.0          171.00
2           2.0          100.50
3           0.0            0.00
4           0.0            0.00
5           0.0            0.00
6           1.0           40.00
7           0.0            0.00
8           0.0            0.00
9           1.0           57.95


In [7]:
start = time.time()

result_1h = compute_rolling_features_vectorized(full_df, 3600, 'txn_1h')
result_24h = compute_rolling_features_vectorized(full_df, 86400, 'txn_24h')

elapsed = time.time() - start
print(f"Full dataset (both windows) time: {elapsed:.1f}s")
print(f"\n1h features shape: {result_1h.shape}")
print(f"24h features shape: {result_24h.shape}")

Full dataset (both windows) time: 21.1s

1h features shape: (590540, 2)
24h features shape: (590540, 2)


In [8]:
full_df = full_df.sort_values(['card_entity', 'TransactionDT']).reset_index(drop=True)

full_df['txn_1h_count'] = result_1h['txn_1h_count'].values
full_df['txn_1h_avg_amt'] = result_1h['txn_1h_avg_amt'].values
full_df['txn_24h_count'] = result_24h['txn_24h_count'].values
full_df['txn_24h_avg_amt'] = result_24h['txn_24h_avg_amt'].values

# Now sort back to chronological order for the actual train/val split
full_df = full_df.sort_values('TransactionDT').reset_index(drop=True)

print(full_df.shape)
print(full_df[['card_entity', 'TransactionDT', 'entity_txn_count_so_far', 'time_since_last_txn', 
               'txn_1h_count', 'txn_1h_avg_amt', 'txn_24h_count', 'txn_24h_avg_amt']].head(10))

/tmp/ipykernel_1974/1449879539.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['txn_1h_count'] = result_1h['txn_1h_count'].values
/tmp/ipykernel_1974/1449879539.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['txn_1h_avg_amt'] = result_1h['txn_1h_avg_amt'].values
/tmp/ipykernel_1974/1449879539.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) i

(590540, 441)
         card_entity  TransactionDT  entity_txn_count_so_far  \
0  13926_361.0_315.0          86400                        0   
1   2755_404.0_325.0          86401                        0   
2   4663_490.0_330.0          86469                        0   
3  18132_567.0_476.0          86499                        0   
4   4497_514.0_420.0          86506                        0   
5   5937_555.0_272.0          86510                        0   
6  12308_360.0_126.0          86522                        0   
7  12695_490.0_325.0          86529                        0   
8   2803_100.0_337.0          86535                        0   
9  17399_111.0_204.0          86536                        0   

   time_since_last_txn  txn_1h_count  txn_1h_avg_amt  txn_24h_count  \
0             999999.0           0.0             0.0            0.0   
1             999999.0           0.0             0.0            0.0   
2             999999.0           0.0             0.0            0.0 

In [9]:
non_zero_1h = (full_df['txn_1h_count'] > 0).sum()
non_zero_24h = (full_df['txn_24h_count'] > 0).sum()

print(f"Rows with txn_1h_count > 0: {non_zero_1h} ({non_zero_1h/len(full_df)*100:.1f}%)")
print(f"Rows with txn_24h_count > 0: {non_zero_24h} ({non_zero_24h/len(full_df)*100:.1f}%)")

print("\ntxn_1h_count distribution:")
print(full_df['txn_1h_count'].describe())

print("\ntxn_24h_count distribution:")
print(full_df['txn_24h_count'].describe())

Rows with txn_1h_count > 0: 146191 (24.8%)
Rows with txn_24h_count > 0: 345089 (58.4%)

txn_1h_count distribution:
count    590540.000000
mean          0.646281
std           4.687272
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         191.000000
Name: txn_1h_count, dtype: float64

txn_24h_count distribution:
count    590540.000000
mean          5.945465
std          24.028783
min           0.000000
25%           0.000000
50%           1.000000
75%           4.000000
max         880.000000
Name: txn_24h_count, dtype: float64


In [10]:
print("Fraud rate by txn_1h_count buckets:")
full_df['txn_1h_bucket'] = pd.cut(full_df['txn_1h_count'], bins=[-1, 0, 1, 3, 10, 1000], labels=['0', '1', '2-3', '4-10', '10+'])
print(full_df.groupby('txn_1h_bucket', observed=True)['isFraud'].agg(['mean', 'count']))

print("\nFraud rate by txn_24h_count buckets:")
full_df['txn_24h_bucket'] = pd.cut(full_df['txn_24h_count'], bins=[-1, 0, 1, 3, 10, 1000], labels=['0', '1', '2-3', '4-10', '10+'])
print(full_df.groupby('txn_24h_bucket', observed=True)['isFraud'].agg(['mean', 'count']))

Fraud rate by txn_1h_count buckets:
                   mean   count
txn_1h_bucket                  
0              0.028761  444349
1              0.049230   85659
2-3            0.059457   43443
4-10           0.066065   14970
10+            0.044361    2119

Fraud rate by txn_24h_count buckets:
                    mean   count
txn_24h_bucket                  
0               0.023817  245451
1               0.034803   91114
2-3             0.041692   83254
4-10            0.048219   91189
10+             0.047503   79532


/tmp/ipykernel_1974/2150439420.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['txn_1h_bucket'] = pd.cut(full_df['txn_1h_count'], bins=[-1, 0, 1, 3, 10, 1000], labels=['0', '1', '2-3', '4-10', '10+'])
/tmp/ipykernel_1974/2150439420.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['txn_24h_bucket'] = pd.cut(full_df['txn_24h_count'], bins=[-1, 0, 1, 3, 10, 1000], labels=['0', '1', '2-3', '4-10', '10+'])


In [11]:
full_df = full_df.drop(columns=['txn_1h_bucket', 'txn_24h_bucket'])
print(full_df.shape)

(590540, 441)


In [12]:
full_df = full_df.sort_values('TransactionDT').reset_index(drop=True)

train_df_v2 = full_df.iloc[:split_point].copy()
val_df_v2 = full_df.iloc[split_point:].copy()

print(f"Train: {train_df_v2.shape}, Val: {val_df_v2.shape}")
print(f"\nTrain date range: {train_df_v2['TransactionDT'].min()} to {train_df_v2['TransactionDT'].max()}")
print(f"Val date range: {val_df_v2['TransactionDT'].min()} to {val_df_v2['TransactionDT'].max()}")

# Sanity check: should exactly match Phase 2's split
print(f"\nMatches original split point ({split_point})? {len(train_df_v2) == split_point}")

Train: (472432, 441), Val: (118108, 441)

Train date range: 86400 to 12192842
Val date range: 12192900 to 15811131

Matches original split point (472432)? True


In [13]:
import xgboost as xgb
import time

exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT', 'card_entity']
feature_cols_v2 = [c for c in full_df.columns if c not in exclude_cols]

X_train_v2 = train_df_v2[feature_cols_v2]
y_train_v2 = train_df_v2['isFraud']
X_val_v2 = val_df_v2[feature_cols_v2]
y_val_v2 = val_df_v2['isFraud']

print(f"Feature count: {len(feature_cols_v2)} (was 431 before)")

scale_pos_weight_v2 = (y_train_v2 == 0).sum() / (y_train_v2 == 1).sum()

xgb_model_v3 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_v2,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

start = time.time()
xgb_model_v3.fit(X_train_v2, y_train_v2)
print(f"Training time: {time.time() - start:.1f}s")

Feature count: 437 (was 431 before)
Training time: 13.0s


In [14]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, precision_recall_curve

y_pred_proba_v3 = xgb_model_v3.predict_proba(X_val_v2)[:, 1]

auc_v3 = roc_auc_score(y_val_v2, y_pred_proba_v3)
print(f"AUC-ROC (with velocity features): {auc_v3:.4f}")
print(f"AUC-ROC (baseline, Phase 3):      0.9026")
print(f"Improvement: {auc_v3 - 0.9026:+.4f}")

precisions_v3, recalls_v3, thresholds_v3 = precision_recall_curve(y_val_v2, y_pred_proba_v3)

f2_scores_v3 = 5 * (precisions_v3 * recalls_v3) / (4 * precisions_v3 + recalls_v3 + 1e-10)
best_f2_idx_v3 = np.argmax(f2_scores_v3)

print(f"\nBest F2 threshold: {thresholds_v3[best_f2_idx_v3]:.4f}")
print(f"At this threshold -> Precision: {precisions_v3[best_f2_idx_v3]:.4f}, Recall: {recalls_v3[best_f2_idx_v3]:.4f}, F2: {f2_scores_v3[best_f2_idx_v3]:.4f}")
print(f"Baseline F2 (Phase 3): 0.5414")
print(f"Improvement: {f2_scores_v3[best_f2_idx_v3] - 0.5414:+.4f}")

AUC-ROC (with velocity features): 0.9066
AUC-ROC (baseline, Phase 3):      0.9026
Improvement: +0.0040

Best F2 threshold: 0.6210
At this threshold -> Precision: 0.3133, Recall: 0.6398, F2: 0.5294
Baseline F2 (Phase 3): 0.5414
Improvement: -0.0120


In [15]:
importance = pd.DataFrame({
    'feature': feature_cols_v2,
    'importance': xgb_model_v3.feature_importances_
}).sort_values('importance', ascending=False)

new_features = ['entity_txn_count_so_far', 'time_since_last_txn', 'txn_1h_count', 'txn_1h_avg_amt', 'txn_24h_count', 'txn_24h_avg_amt']

print("Top 20 most important features overall:")
print(importance.head(20))

print("\nRanking of our new velocity features specifically:")
for feat in new_features:
    rank = importance.reset_index(drop=True)
    idx = rank[rank['feature'] == feat].index[0]
    print(f"{feat}: rank {idx+1} of {len(importance)}, importance = {rank.loc[idx, 'importance']:.5f}")

Top 20 most important features overall:
         feature  importance
293         V258    0.125572
105          V70    0.079194
126          V91    0.068058
329         V294    0.049024
236         V201    0.026203
402  card6_debit    0.020423
222         V187    0.015649
15            C8    0.014337
21           C14    0.013383
390  ProductCD_C    0.012749
11            C4    0.012531
163         V128    0.012319
343         V308    0.012063
84           V49    0.011810
347         V312    0.010932
410      M4_freq    0.008983
18           C11    0.007599
80           V45    0.007256
23            D2    0.006744
137         V102    0.005990

Ranking of our new velocity features specifically:
entity_txn_count_so_far: rank 128 of 437, importance = 0.00161
time_since_last_txn: rank 146 of 437, importance = 0.00144
txn_1h_count: rank 255 of 437, importance = 0.00074
txn_1h_avg_amt: rank 273 of 437, importance = 0.00062
txn_24h_count: rank 235 of 437, importance = 0.00086
txn_24h_avg_amt: r

In [16]:
import joblib
import os

joblib.dump(train_df_v2, '../data/processed/train_processed_v2.pkl', compress=3)
joblib.dump(val_df_v2, '../data/processed/val_processed_v2.pkl', compress=3)

print(f"train_processed_v2.pkl: {os.path.getsize('../data/processed/train_processed_v2.pkl') / 1e6:.1f} MB")
print(f"val_processed_v2.pkl: {os.path.getsize('../data/processed/val_processed_v2.pkl') / 1e6:.1f} MB")

train_processed_v2.pkl: 68.2 MB
val_processed_v2.pkl: 17.6 MB


In [17]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(xgb_model_v3, '../models/xgb_fraud_model_v1.pkl')

print(f"Model saved: {os.path.getsize('../models/xgb_fraud_model_v1.pkl') / 1e6:.1f} MB")

Model saved: 0.8 MB
